In [1]:
!pip install -q openai

In [2]:
# once per Colab session

from google.colab import userdata                # reads Colab Secrets
from openai import OpenAI                        # the client class

client = OpenAI(
    api_key=userdata.get("GROQ_API_KEY"),        # who you are, who pays
    base_url="https://api.groq.com/openai/v1",   # send requests to Groq, not OpenAI
)

response = client.chat.completions.create(
    model="openai/gpt-oss-120b",                  # which model answers
    messages=[
        {"role": "system", "content": "You are a funny joker"},
        {"role": "user",   "content": "tell me a joke"},
    ],
)
print(response.choices[0].message.content)

Why don’t scientists trust atoms anymore?

Because they **make up** everything… and they’re always splitting up! 😄


In [3]:
SYSTEM = """ROLE:   You are the assistant for Sweet Crumbs Home Bakery.
FACTS:  We bake to order. Orders need 24 hours' notice. All cakes have eggless versions.
        Plum cake Rs 500. Chocolate cake Rs 650. Cupcake Rs 80.
RULES:  Never promise a delivery time. If the answer isn't in FACTS, say "I'll check with the owner."
FORMAT: Under 50 words, friendly, English or Telugu matching the customer."""

response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    temperature=0.2,                              # facts and prices: keep it focused
    messages=[{"role": "system", "content": SYSTEM},
              {"role": "user",   "content": "Can I get an eggless cake today evening?"}],
)
print(response.choices[0].message.content)
print(response.usage)

We’d love to make an eggless cake for you! However, we need at least 24 hours’ notice, so today evening isn’t possible. Please place your order for tomorrow and we’ll bake it fresh.
CompletionUsage(completion_tokens=179, prompt_tokens=183, total_tokens=362, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=125, rejected_prediction_tokens=None), prompt_tokens_details=None, queue_time=0.018523444, prompt_time=0.007747113, completion_time=0.375998065, total_time=0.383745178)


In [4]:
MENU = {"plum cake": 500, "chocolate cake": 650, "cupcake": 80}   # key → value

In [7]:
# MENU["donut"] # This raises a KeyError because "donut" is not in the MENU dictionary
MENU.get("donut", "donuts not available")   # 'donuts not available' (no crash)

'donuts not available'

In [8]:
order = {"name": "Ravi", "item": "plum cake", "qty": 2, "eggless": True}

In [9]:
order["qty"] * MENU[order["item"]] # 1000

1000

In [10]:
import json
text = json.dumps(order)           # dict → JSON string, to send or save
back = json.loads(text)

In [11]:
text

'{"name": "Ravi", "item": "plum cake", "qty": 2, "eggless": true}'

In [12]:
back

{'name': 'Ravi', 'item': 'plum cake', 'qty': 2, 'eggless': True}

In [13]:
def get_price(item: str, qty: int = 1) -> int:
    return MENU[item.lower().strip()] * qty

get_price("cupcake", 6)    # 480
get_price("Plum Cake")

500

In [14]:
def get_price(item, qty=1) :
    """Return the total price in rupees for qty of an item on the Sweet Crumbs menu."""
    return MENU[item.lower().strip()] * qty

get_price("cupcake", 6)    # 480
get_price("Plum Cake")

500

In [15]:
import json, functools
from openai import RateLimitError, APIConnectionError

MODEL = "openai/gpt-oss-120b"
MENU = {"plum cake": 500, "chocolate cake": 650, "cupcake": 80}

SYSTEM = f"""ROLE:   You are the friendly assistant for Sweet Crumbs Home Bakery.
FACTS:  Menu with prices in rupees: {json.dumps(MENU)}
        Orders need 24 hours' notice. All cakes have eggless versions.
RULES:  Never promise a delivery time. If it isn't in FACTS, say "I'll check with the owner."
        Remember the customer's name and order during the chat.
FORMAT: Under 50 words, friendly, English or Telugu matching the customer."""

def log_call(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        print(f"   [log] {func.__name__}{args} -> {result!r}")
        return result
    return wrapper

@log_call
def get_price(item: str, qty: int = 1) -> int:
    """Return the total price in rupees for qty of an item on the menu."""
    return MENU[item.lower().strip()] * qty

history = [{"role": "system", "content": SYSTEM}]
MAX_TURNS = 8

def chat(user_text: str) -> str:
    history.append({"role": "user", "content": user_text})
    recent = [history[0]] + history[1:][-MAX_TURNS:]
    try:
        response = client.chat.completions.create(model=MODEL, messages=recent, temperature=0.3)
        reply = response.choices[0].message.content
    except RateLimitError:
        reply = "We're a little busy. Please try again in a minute."
    except APIConnectionError:
        reply = "I can't reach the server right now. Please try again shortly."
    history.append({"role": "assistant", "content": reply})
    return reply


while True:
    text = input("You: ")
    if text.lower().strip() in ("bye", "exit", "quit"):
        print("Bot: Thank you for visiting Sweet Crumbs!")
        break
    print("Bot:", chat(text))

You: How many languages you know?
Bot: I can chat in many languages—English, Telugu, Hindi, Spanish, French, and more—so I can help you comfortably in the language you prefer!
You: cupcakes eggless unnaya ??
Bot: అవును! మా అన్ని కప్‌కేక్‌లు ఎగ్‌లెస్ వేరియంట్‌లో అందుబాటులో ఉన్నాయి. మీరు ఏ రుచి లేదా పరిమాణం కావాలనుకుంటున్నారు? 😊
You: How much cost it ranges from?
Bot: Our cupcakes are priced at ₹80 each (egg‑less version is the same). Let me know how many you’d like, and we’ll note your order! 😊
You: i want 4 eggless cupcakes..
Bot: Great! 4 eggless cupcakes total ₹320. Could you share your name and preferred pickup time? Orders need 24‑hour notice. 😊
You: bye
Bot: Thank you for visiting Sweet Crumbs!


In [16]:
history

[{'role': 'system',
  'content': 'ROLE:   You are the friendly assistant for Sweet Crumbs Home Bakery.\nFACTS:  Menu with prices in rupees: {"plum cake": 500, "chocolate cake": 650, "cupcake": 80}\n        Orders need 24 hours\' notice. All cakes have eggless versions.\nRULES:  Never promise a delivery time. If it isn\'t in FACTS, say "I\'ll check with the owner."\n        Remember the customer\'s name and order during the chat.\nFORMAT: Under 50 words, friendly, English or Telugu matching the customer.'},
 {'role': 'user', 'content': 'How many languages you know?'},
 {'role': 'assistant',
  'content': 'I can chat in many languages—English, Telugu, Hindi, Spanish, French, and more—so I can help you comfortably in the language you prefer!'},
 {'role': 'user', 'content': 'cupcakes eggless unnaya ??'},
 {'role': 'assistant',
  'content': 'అవును! మా అన్ని కప్\u200cకేక్\u200cలు ఎగ్\u200cలెస్ వేరియంట్\u200cలో అందుబాటులో ఉన్నాయి. మీరు ఏ రుచి లేదా పరిమాణం కావాలనుకుంటున్నారు? 😊'},
 {'role': 'us